<a href="https://colab.research.google.com/github/queirozcleiton/analise_investimentos/blob/main/analisar_fiis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
# ============================================================
# NOME DO SCRIPT: analisar_fiis_colab.py
# OBJETIVO: Coletar dados de FIIs do Fundamentus, aplicar filtros
#           de qualidade previdenciária e exportar planilha Excel
#           formatada com ranking multicritério
# BIBLIOTECAS NECESSÁRIAS: requests, pandas, openpyxl, lxml
# COMO EXECUTAR: Google Colab — rodar célula por célula
# ============================================================

# ===========================================================
# CÉLULA 1 — Instalação das bibliotecas
# ===========================================================
# No Colab, algumas bibliotecas não vêm pré-instaladas.
# Esta célula garante que tudo está disponível antes de começar.
# Execute esta célula primeiro e aguarde a conclusão.

!pip install requests pandas openpyxl lxml --quiet

In [34]:
# @title
# CÉLULA DE DIAGNÓSTICO — rode antes de executar_analise()

#df_teste = coletar_fiis_fundamentus()

#if df_teste is not None:
 #   print(f"\nTotal de linhas: {len(df_teste)}")
 #   print(f"\nColunas encontradas:")
 #   for i, col in enumerate(df_teste.columns):
 #       print(f"  [{i}] '{col}'")
 #   print(f"\nPrimeiras 3 linhas:")
 #   print(df_teste.head(3).to_string())

In [35]:
# ===========================================================
# CÉLULA 2 — Importações
# ===========================================================

# --- Bibliotecas padrão do Python (já vêm instaladas) ---
import time          # Para pausas entre requisições (boa prática de scraping)
import re            # Para expressões regulares (limpeza de texto)
from datetime import datetime  # Para registrar data/hora da coleta
import io            # Para manipular dados em memória (download no Colab)

# --- Bibliotecas de dados ---
import requests      # Para fazer requisições HTTP ao Fundamentus
import pandas as pd  # Para manipular tabelas de dados (DataFrames)

# --- Biblioteca para gerar o arquivo Excel ---
from openpyxl import Workbook                          # Criar planilha
from openpyxl.styles import (
    Font,            # Controlar fonte (negrito, cor, tamanho)
    PatternFill,     # Cor de fundo das células
    Alignment,       # Alinhamento do texto
    Border,          # Bordas das células
    Side             # Estilo das linhas de borda
)
from openpyxl.utils import get_column_letter  # Converte número de coluna → letra (1 → 'A')

# --- Para download automático no Colab ---
try:
    # Esta função só existe no ambiente do Google Colab
    from google.colab import files
    RODANDO_NO_COLAB = True
except ImportError:
    # Se não estiver no Colab, salva localmente sem erro
    RODANDO_NO_COLAB = False

print("✅ Bibliotecas importadas com sucesso!")
print(f"📅 Data/hora da análise: {datetime.now().strftime('%d/%m/%Y às %H:%M')}")

✅ Bibliotecas importadas com sucesso!
📅 Data/hora da análise: 17/06/2026 às 20:06


In [36]:
# ===========================================================
# CÉLULA 3 — Configurações (ajuste aqui conforme necessário)
# ===========================================================

# --- Filtros de qualidade mínima ---
# FIIs que não atingirem esses critérios serão excluídos da planilha.
# Você pode afrouxar ou apertar esses valores conforme sua estratégia.

FILTRO_LIQUIDEZ_MINIMA    = 500_000   # R$ 500 mil de volume médio diário
                                       # Garante que você consegue comprar/vender sem impactar o preço

FILTRO_PVP_MINIMO         = 0.00      # P/VP mínimo
FILTRO_PVP_MAXIMO         = 99.0      # P/VP máximo
                                       # Faixa razoável: nem muito barato (sinal de problema)
                                       # nem muito caro (prêmio excessivo sobre o patrimônio)

FILTRO_DY_MINIMO          = 0.05      # DY mínimo de 5% ao ano
                                       # Abaixo disso, o rendimento não compensa o risco

FILTRO_VACANCIA_MAXIMA    = 0.30      # Vacância máxima de 30%
                                       # Acima disso, muitos imóveis vazios = risco de corte de rendimento

# --- Pesos do ranking multicritério ---
# A soma dos pesos deve ser 1.0 (100%).
# Você pode redistribuir conforme sua preferência.
PESO_DY                   = 0.30      # 30% — Rendimento é o foco previdenciário
PESO_PVP                  = 0.25      # 25% — Valuation: quanto estou pagando pelo patrimônio
PESO_VACANCIA             = 0.25      # 25% — Qualidade dos imóveis (vacância baixa = imóveis ocupados)
PESO_LIQUIDEZ             = 0.20      # 20% — Facilidade de operar sem impactar o preço

# --- Nome do arquivo de saída ---
NOME_ARQUIVO = f"analise_fiis_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"

print("✅ Configurações definidas!")
print(f"\n📋 Filtros ativos:")
print(f"   Liquidez mínima:  R$ {FILTRO_LIQUIDEZ_MINIMA:,.0f}")
print(f"   P/VP:             entre {FILTRO_PVP_MINIMO} e {FILTRO_PVP_MAXIMO}")
print(f"   DY mínimo:        {FILTRO_DY_MINIMO*100:.0f}% ao ano")
print(f"   Vacância máxima:  {FILTRO_VACANCIA_MAXIMA*100:.0f}%")

✅ Configurações definidas!

📋 Filtros ativos:
   Liquidez mínima:  R$ 500,000
   P/VP:             entre 0.0 e 99.0
   DY mínimo:        5% ao ano
   Vacância máxima:  30%


In [37]:
# ===========================================================
# CÉLULA 4 — Funções auxiliares
# ===========================================================

def converter_percentual_br(valor):
    """
    Converte string de percentual brasileiro para float decimal.

    Exemplos:
        "14,05%"  →  0.1405
        "3,91%"   →  0.0391
        0.86      →  0.86   (já é float, retorna sem alterar)
    """
    if pd.isna(valor):
        return None
    if isinstance(valor, (int, float)):
        return float(valor)
    texto = str(valor).strip().replace('%', '').replace('.', '').replace(',', '.')
    try:
        return float(texto) / 100
    except ValueError:
        return None


def converter_numero_br(valor):
    """
    Converte string numérica no formato brasileiro para float.

    Exemplos:
        "1.234,56"  →  1234.56
        "74,81"     →  74.81
        74.81       →  74.81  (já é float, retorna sem alterar)
    """
    if pd.isna(valor):
        return None
    if isinstance(valor, (int, float)):
        return float(valor)
    texto = str(valor).strip().replace('.', '').replace(',', '.')
    try:
        return float(texto)
    except ValueError:
        return None


def converter_pvp(valor):
    """
    Converte o P/VP do Fundamentus para float na escala correta.

    O Fundamentus atualmente entrega o P/VP como número inteiro
    sem vírgula decimal. Exemplos observados:
        24   →  0.24   (significa P/VP de 0,24)
        93   →  0.93
        100  →  1.00
        130  →  1.30

    A regra é: se o valor for >= 2, divide por 100.
    Valores < 2 já estão na escala correta (ex: 1.06 = P/VP de 1,06).

    Parâmetros:
        valor: int, float ou string

    Retorna:
        float ou None
    """
    if pd.isna(valor):
        return None

    # Converte para float primeiro (trata strings com separadores BR)
    if isinstance(valor, str):
        texto = valor.strip().replace('.', '').replace(',', '.')
        try:
            numero = float(texto)
        except ValueError:
            return None
    else:
        numero = float(valor)

    # P/VP >= 2 indica que está na escala errada (inteiro sem decimal)
    # Nenhum FII saudável tem P/VP real acima de 2,0 de forma consistente
    if numero >= 2:
        return numero / 100

    return numero


def classificar_tipo_fii(segmento):
    """
    Classifica o FII em uma das quatro categorias principais
    com base no segmento informado pelo Fundamentus.

    Categorias:
        Tijolo  — imóveis físicos (logística, shopping, escritório, etc.)
        Papel   — títulos de crédito imobiliário (CRI, recebíveis)
        FOF     — fundos que investem em cotas de outros FIIs
        Híbrido — combinação de estratégias ou não classificado

    Parâmetros:
        segmento: string com o segmento do Fundamentus

    Retorna:
        string com o tipo do FII
    """
    if pd.isna(segmento):
        return 'Híbrido'

    seg = str(segmento).lower()

    # FIIs de Papel investem em títulos de crédito imobiliário
    if any(termo in seg for termo in ['títulos', 'val. mob', 'recebív', 'cri']):
        return 'Papel'

    # FOFs investem em cotas de outros FIIs
    if 'fundo de fundos' in seg or 'fof' in seg:
        return 'FOF'

    # FIIs de Tijolo possuem imóveis físicos
    if any(termo in seg for termo in [
        'logística', 'shopping', 'lajes', 'escritório', 'corporativ',
        'industrial', 'varejo', 'hospital', 'educacion', 'hotel',
        'residencial', 'agro'
    ]):
        return 'Tijolo'

    # Multicategoria e Outros entram como Híbrido
    return 'Híbrido'


def coletar_fiis_fundamentus():
    """
    Coleta a tabela completa de FIIs do site Fundamentus.

    Retorna:
        DataFrame com todos os FIIs e indicadores brutos,
        ou None em caso de falha na coleta.
    """

    url = "https://www.fundamentus.com.br/fii_resultado.php"

    headers = {
        'User-Agent': (
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/120.0.0.0 Safari/537.36'
        ),
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Accept-Language': 'pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7',
        'Referer': 'https://www.fundamentus.com.br/'
    }

    print("🌐 Conectando ao Fundamentus...")

    try:
        resposta = requests.get(url, headers=headers, timeout=30)
        resposta.raise_for_status()

        print("✅ Conexão bem-sucedida! Extraindo tabela...")

        # Não passamos decimal/thousands — percentuais chegam como string
        # e serão convertidos manualmente pelas funções auxiliares
        tabelas = pd.read_html(io.StringIO(resposta.text))
        df = tabelas[0]

        print(f"📊 {len(df)} FIIs encontrados no Fundamentus.")
        return df

    except requests.exceptions.ConnectionError:
        print("❌ Erro: Sem conexão com a internet.")
        return None
    except requests.exceptions.Timeout:
        print("❌ Erro: O Fundamentus demorou demais para responder.")
        return None
    except requests.exceptions.HTTPError as e:
        print(f"❌ Erro HTTP: {e}")
        return None
    except Exception as e:
        print(f"❌ Erro inesperado na coleta: {e}")
        return None


def limpar_e_padronizar(df_bruto):
    """
    Limpa e padroniza o DataFrame bruto do Fundamentus.

    Etapas:
    1. Renomeia colunas para nomes padronizados em português
    2. Converte percentuais (string "14,05%" → float 0.1405)
    3. Converte P/VP com tratamento especial (inteiro → decimal)
    4. Converte demais números
    5. Classifica cada FII por tipo (Tijolo, Papel, FOF, Híbrido)
    6. Remove linhas com dados críticos faltando
    7. Remove fundos fantasmas (cotação, liquidez e DY todos zero)

    Parâmetros:
        df_bruto: DataFrame retornado diretamente pelo read_html()

    Retorna:
        DataFrame limpo, padronizado e com coluna 'tipo_fii'
    """

    print("\n🔧 Limpando e padronizando os dados...")

    mapa_colunas = {
        'Papel':            'ticker',
        'Segmento':         'segmento',
        'Cotação':          'cotacao',
        'FFO Yield':        'ffo_yield',
        'Dividend Yield':   'dy_anual',
        'P/VP':             'pvp',
        'Valor de Mercado': 'valor_mercado',
        'Liquidez':         'liquidez_diaria',
        'Qtd de imóveis':   'qtd_imoveis',
        'Vacância Média':   'vacancia',
        'Cap Rate':         'cap_rate',
        'Preço do m2':      'preco_m2',
        'Aluguel por m2':   'aluguel_m2',
    }

    colunas_presentes = {k: v for k, v in mapa_colunas.items() if k in df_bruto.columns}
    df = df_bruto.rename(columns=colunas_presentes)

    # Percentuais chegam como string "14,05%" → converter para 0.1405
    colunas_percentual = ['dy_anual', 'ffo_yield', 'vacancia', 'cap_rate']
    for col in colunas_percentual:
        if col in df.columns:
            df[col] = df[col].apply(converter_percentual_br)

    # P/VP tem tratamento especial — pode chegar como inteiro sem decimal
    if 'pvp' in df.columns:
        df['pvp'] = df['pvp'].apply(converter_pvp)

    # Demais colunas numéricas
    colunas_numericas = ['cotacao', 'valor_mercado', 'liquidez_diaria',
                         'qtd_imoveis', 'preco_m2', 'aluguel_m2']
    for col in colunas_numericas:
        if col in df.columns:
            df[col] = df[col].apply(converter_numero_br)

    # Classifica cada FII por tipo com base no segmento
    df['tipo_fii'] = df['segmento'].apply(classificar_tipo_fii)

    # Para FIIs de Papel e FOF, vacância zero faz sentido mesmo sem imóveis
    # Para Tijolo, vacância zero com qtd_imoveis zero pode ser dado ausente
    # Marcamos como NaN para não distorcer o score de vacância
    if 'vacancia' in df.columns and 'qtd_imoveis' in df.columns:
        mascara_tijolo_sem_imovel = (
            (df['tipo_fii'] == 'Tijolo') &
            (df['qtd_imoveis'] == 0) &
            (df['vacancia'] == 0)
        )
        df.loc[mascara_tijolo_sem_imovel, 'vacancia'] = None

    # Remove linhas sem dados críticos
    colunas_obrigatorias = ['ticker', 'dy_anual', 'pvp', 'liquidez_diaria']
    linhas_antes = len(df)
    df = df.dropna(subset=colunas_obrigatorias)
    print(f"   {linhas_antes - len(df)} FIIs removidos por dados faltantes.")

    # Remove fundos fantasmas (sem cotação, liquidez e DY ao mesmo tempo)
    mascara_fantasma = (
        (df['dy_anual']        == 0) &
        (df['liquidez_diaria'] == 0) &
        (df['cotacao']         == 0)
    )
    df = df[~mascara_fantasma]

    # Distribuição por tipo para conferência
    print(f"   {len(df)} FIIs com dados válidos.")
    print(f"\n   Distribuição por tipo:")
    for tipo, contagem in df['tipo_fii'].value_counts().items():
        print(f"      {tipo}: {contagem} FIIs")

    return df.reset_index(drop=True)


def aplicar_filtros_qualidade(df):
    """
    Aplica os filtros de qualidade mínima definidos nas configurações.

    Parâmetros:
        df: DataFrame limpo e padronizado

    Retorna:
        DataFrame filtrado com apenas FIIs que passaram em todos os critérios
    """

    print("\n🔍 Aplicando filtros de qualidade...")
    total_inicial = len(df)

    # Filtro 1: Liquidez mínima
    if 'liquidez_diaria' in df.columns:
        antes = len(df)
        df = df[df['liquidez_diaria'] >= FILTRO_LIQUIDEZ_MINIMA]
        print(f"   Liquidez < R$ {FILTRO_LIQUIDEZ_MINIMA:,.0f}: {antes - len(df)} removidos")

    # Filtro 2: P/VP dentro da faixa (agora na escala correta 0.0–2.0)
    if 'pvp' in df.columns:
        antes = len(df)
        df = df[(df['pvp'] >= FILTRO_PVP_MINIMO) & (df['pvp'] <= FILTRO_PVP_MAXIMO)]
        print(f"   P/VP fora de [{FILTRO_PVP_MINIMO}, {FILTRO_PVP_MAXIMO}]: {antes - len(df)} removidos")

    # Filtro 3: DY mínimo
    if 'dy_anual' in df.columns:
        antes = len(df)
        df = df[df['dy_anual'] >= FILTRO_DY_MINIMO]
        print(f"   DY < {FILTRO_DY_MINIMO*100:.0f}%: {antes - len(df)} removidos")

    # Filtro 4: Vacância máxima
    # FIIs de Papel e FOF não têm vacância (NaN) e são mantidos
    if 'vacancia' in df.columns:
        antes = len(df)
        df = df[(df['vacancia'].isna()) | (df['vacancia'] <= FILTRO_VACANCIA_MAXIMA)]
        print(f"   Vacância > {FILTRO_VACANCIA_MAXIMA*100:.0f}%: {antes - len(df)} removidos")

    print(f"\n✅ {len(df)} FIIs aprovados (de {total_inicial} analisados)")
    return df.reset_index(drop=True)


def calcular_score(df):
    """
    Calcula o score multicritério de cada FII aprovado nos filtros.

    Normalização min-max dentro de cada grupo de tipo de FII:
    cada indicador é convertido para escala 0–10, onde o melhor
    valor do grupo recebe 10 e o pior recebe 0.

    O score final é a média ponderada dos scores individuais.

    Parâmetros:
        df: DataFrame filtrado

    Retorna:
        DataFrame com scores calculados, ordenado por score_final
    """

    print("\n📐 Calculando scores do ranking...")

    df = df.copy()

    def normalizar(serie, inverter=False):
        """Normaliza para escala 0–10. inverter=True para indicadores
        onde menor valor é melhor (vacância, distância do P/VP a 1,0)."""
        minimo = serie.min()
        maximo = serie.max()
        if maximo == minimo:
            return pd.Series([5.0] * len(serie), index=serie.index)
        normalizado = (serie - minimo) / (maximo - minimo) * 10
        return (10 - normalizado) if inverter else normalizado

    # Score DY: maior → melhor
    df['score_dy'] = normalizar(df['dy_anual']) if 'dy_anual' in df.columns else 5.0

    # Score P/VP: mais próximo de 1,0 → melhor
    if 'pvp' in df.columns:
        df['score_pvp'] = normalizar((df['pvp'] - 1.0).abs(), inverter=True)
    else:
        df['score_pvp'] = 5.0

    # Score Vacância: menor → melhor
    # NaN (Papel/FOF) recebe 0 (sem vacância = melhor possível)
    if 'vacancia' in df.columns:
        df['score_vacancia'] = normalizar(df['vacancia'].fillna(0), inverter=True)
    else:
        df['score_vacancia'] = 5.0

    # Score Liquidez: maior → melhor
    df['score_liquidez'] = normalizar(df['liquidez_diaria']) if 'liquidez_diaria' in df.columns else 5.0

    # Score Final ponderado
    df['score_final'] = (
        df['score_dy']       * PESO_DY       +
        df['score_pvp']      * PESO_PVP      +
        df['score_vacancia'] * PESO_VACANCIA +
        df['score_liquidez'] * PESO_LIQUIDEZ
    ).round(2)

    df = df.sort_values('score_final', ascending=False).reset_index(drop=True)
    df.insert(0, 'ranking', range(1, len(df) + 1))

    print(f"✅ Score calculado para {len(df)} FIIs.")
    return df

In [38]:
# ===========================================================
# CÉLULA 5 — Funções de formatação da planilha Excel
# ===========================================================

def criar_estilo_cabecalho():
    """Retorna configurações de estilo para o cabeçalho da tabela."""
    return {
        'font':      Font(name='Arial', bold=True, color='FFFFFF', size=11),
        'fill':      PatternFill('solid', start_color='1F4E79'),
        'alignment': Alignment(horizontal='center', vertical='center', wrap_text=True),
        'border':    Border(
            left=Side(style='thin', color='FFFFFF'),
            right=Side(style='thin', color='FFFFFF'),
            bottom=Side(style='medium', color='FFFFFF')
        )
    }


def cor_score(score):
    """Cor de fundo para a célula de score final."""
    if score >= 7.0:
        return '70AD47'   # Verde — excelente
    elif score >= 5.0:
        return 'FFC000'   # Amarelo — bom
    else:
        return 'FF6B35'   # Laranja — aceitável


# Definição das colunas da planilha
# Tupla: (nome no DataFrame, título na planilha, largura, formato numérico)
COLUNAS_PLANILHA = [
    ('ranking',         '#',             6,   None),
    ('ticker',          'Ticker',        12,  None),
    ('tipo_fii',        'Tipo',          10,  None),
    ('segmento',        'Segmento',      22,  None),
    ('cotacao',         'Cotação (R$)',  14,  'R$ #,##0.00'),
    ('dy_anual',        'DY Anual',      12,  '0.00%'),
    ('pvp',             'P/VP',          10,  '0.00'),
    ('vacancia',        'Vacância',      12,  '0.00%'),
    ('liquidez_diaria', 'Liquidez/dia',  16,  'R$ #,##0'),
    ('qtd_imoveis',     'Qtd Imóveis',  13,  '0'),
    ('score_dy',        'Score DY',      11,  '0.0'),
    ('score_pvp',       'Score P/VP',    12,  '0.0'),
    ('score_vacancia',  'Score Vac.',    12,  '0.0'),
    ('score_liquidez',  'Score Liq.',    12,  '0.0'),
    ('score_final',     'Score Final',   12,  '0.00'),
]

# Cores de cabeçalho por tipo de FII para diferenciar as abas visualmente
COR_CABECALHO_TIPO = {
    'Geral':   '1F4E79',   # Azul escuro
    'Tijolo':  '833C00',   # Marrom — imóveis físicos
    'Papel':   '375623',   # Verde escuro — títulos
    'FOF':     '4A235A',   # Roxo — fundo de fundos
    'Híbrido': '1F3864',   # Azul médio — outros
}


def escrever_aba(aba, df_aba, titulo_planilha, cor_cabecalho):
    """
    Preenche uma aba do Excel com os dados do DataFrame fornecido.

    Esta função é reutilizada para todas as abas (Geral, Tijolo,
    Papel, FOF, Híbrido), alterando apenas o título e a cor do cabeçalho.

    Parâmetros:
        aba:              objeto Worksheet do openpyxl
        df_aba:           DataFrame com os FIIs desta aba
        titulo_planilha:  texto da linha de título no topo
        cor_cabecalho:    cor hexadecimal do cabeçalho (sem '#')
    """

    num_colunas = len(COLUNAS_PLANILHA)
    ultima_letra = get_column_letter(num_colunas)

    # --- Linha de título ---
    aba.merge_cells(f'A1:{ultima_letra}1')
    celula_titulo = aba['A1']
    celula_titulo.value = titulo_planilha
    celula_titulo.font = Font(name='Arial', bold=True, size=13,
                              color=cor_cabecalho)
    celula_titulo.alignment = Alignment(horizontal='center', vertical='center')
    aba.row_dimensions[1].height = 28

    # --- Linha de cabeçalho ---
    estilo_cab = {
        'font':      Font(name='Arial', bold=True, color='FFFFFF', size=10),
        'fill':      PatternFill('solid', start_color=cor_cabecalho),
        'alignment': Alignment(horizontal='center', vertical='center',
                               wrap_text=True),
        'border':    Border(
            left=Side(style='thin', color='FFFFFF'),
            right=Side(style='thin', color='FFFFFF'),
            bottom=Side(style='medium', color='FFFFFF')
        )
    }
    aba.row_dimensions[2].height = 32

    for col_idx, (_, titulo, largura, _) in enumerate(COLUNAS_PLANILHA, start=1):
        letra = get_column_letter(col_idx)
        cel = aba[f'{letra}2']
        cel.value = titulo
        cel.font      = estilo_cab['font']
        cel.fill      = estilo_cab['fill']
        cel.alignment = estilo_cab['alignment']
        cel.border    = estilo_cab['border']
        aba.column_dimensions[letra].width = largura

    # --- Linhas de dados ---
    # Renumera o ranking localmente dentro de cada aba
    df_local = df_aba.copy().reset_index(drop=True)
    df_local['ranking'] = range(1, len(df_local) + 1)

    for idx_linha, (_, linha) in enumerate(df_local.iterrows()):
        linha_excel = idx_linha + 3
        cor_fundo = 'EBF3FB' if idx_linha % 2 == 0 else 'FFFFFF'

        estilo_linha = {
            'fill':      PatternFill('solid', start_color=cor_fundo),
            'alignment': Alignment(horizontal='center', vertical='center'),
            'font':      Font(name='Arial', size=10),
            'border':    Border(
                left=Side(style='thin', color='BDD7EE'),
                right=Side(style='thin', color='BDD7EE'),
                bottom=Side(style='thin', color='BDD7EE')
            )
        }

        for col_idx, (nome_col, _, _, formato) in enumerate(COLUNAS_PLANILHA, start=1):
            letra = get_column_letter(col_idx)
            cel = aba[f'{letra}{linha_excel}']
            valor = linha.get(nome_col, '')

            # Trata nulos para não exibir "None" ou "nan"
            eh_nulo = False
            try:
                eh_nulo = pd.isna(valor)
            except (TypeError, ValueError):
                pass

            if eh_nulo:
                cel.value = '-'
            else:
                cel.value = valor
                if formato:
                    cel.number_format = formato

            cel.fill      = estilo_linha['fill']
            cel.alignment = estilo_linha['alignment']
            cel.font      = estilo_linha['font']
            cel.border    = estilo_linha['border']

            # Score Final: fundo colorido por faixa
            if nome_col == 'score_final' and isinstance(valor, (int, float)):
                cel.fill = PatternFill('solid', start_color=cor_score(valor))
                cel.font = Font(name='Arial', bold=True, size=10, color='FFFFFF')
                cel.number_format = '0.00'

            # Ticker em destaque
            if nome_col == 'ticker':
                cel.font = Font(name='Arial', bold=True, size=10,
                                color=cor_cabecalho)
                cel.alignment = Alignment(horizontal='left', vertical='center')

    # Congela título e cabeçalho ao rolar
    aba.freeze_panes = 'A3'

    # Exibe aviso se a aba estiver vazia
    if len(df_local) == 0:
        aba['A3'] = 'Nenhum FII deste tipo passou nos filtros definidos.'
        aba['A3'].font = Font(name='Arial', italic=True, color='888888')


def gerar_planilha_excel(df, nome_arquivo):
    """
    Gera o arquivo Excel com múltiplas abas:

    - Ranking Geral  — todos os FIIs aprovados, ordenados por score
    - Tijolo         — apenas FIIs de imóveis físicos
    - Papel          — apenas FIIs de títulos de crédito (CRI)
    - FOF            — apenas Fundos de Fundos
    - Híbrido        — Multicategoria e outros
    - Metadados      — data, filtros e pesos usados na análise

    Parâmetros:
        df:           DataFrame com os FIIs filtrados e rankeados
        nome_arquivo: nome do arquivo .xlsx a ser gerado
    """

    print(f"\n📝 Gerando planilha Excel: {nome_arquivo}")

    wb = Workbook()

    # --- Aba 1: Ranking Geral ---
    aba_geral = wb.active
    aba_geral.title = "Ranking Geral"
    escrever_aba(
        aba_geral,
        df,
        f"📊 Ranking Geral de FIIs — Análise Previdenciária — {datetime.now().strftime('%d/%m/%Y')}",
        COR_CABECALHO_TIPO['Geral']
    )
    print(f"   ✅ Aba 'Ranking Geral': {len(df)} FIIs")

    # --- Abas por tipo de FII ---
    tipos = ['Tijolo', 'Papel', 'FOF', 'Híbrido']

    titulos_tipo = {
        'Tijolo':  '🏢 FIIs de Tijolo — Imóveis Físicos',
        'Papel':   '📄 FIIs de Papel — Títulos de Crédito Imobiliário',
        'FOF':     '📦 FIIs de Fundos (FOF)',
        'Híbrido': '🔀 FIIs Híbridos e Multicategoria',
    }

    for tipo in tipos:
        # Filtra apenas os FIIs deste tipo e renumera o ranking
        df_tipo = df[df['tipo_fii'] == tipo].copy()

        aba = wb.create_sheet(tipo)
        escrever_aba(
            aba,
            df_tipo,
            f"{titulos_tipo[tipo]} — {datetime.now().strftime('%d/%m/%Y')}",
            COR_CABECALHO_TIPO[tipo]
        )
        print(f"   ✅ Aba '{tipo}': {len(df_tipo)} FIIs")

    # --- Aba final: Metadados ---
    aba_meta = wb.create_sheet("Metadados")

    metadados = [
        ("INFORMAÇÕES DA ANÁLISE", ""),
        ("", ""),
        ("Data e hora da coleta",    datetime.now().strftime('%d/%m/%Y às %H:%M')),
        ("Fonte dos dados",          "Fundamentus.com.br"),
        ("", ""),
        ("FILTROS APLICADOS", ""),
        ("", ""),
        ("Liquidez mínima diária",   f"R$ {FILTRO_LIQUIDEZ_MINIMA:,.0f}"),
        ("P/VP mínimo",              f"{FILTRO_PVP_MINIMO}"),
        ("P/VP máximo",              f"{FILTRO_PVP_MAXIMO}"),
        ("DY anual mínimo",          f"{FILTRO_DY_MINIMO*100:.0f}%"),
        ("Vacância máxima",          f"{FILTRO_VACANCIA_MAXIMA*100:.0f}%"),
        ("", ""),
        ("PESOS DO RANKING", ""),
        ("", ""),
        ("Peso DY",                  f"{PESO_DY*100:.0f}%"),
        ("Peso P/VP",                f"{PESO_PVP*100:.0f}%"),
        ("Peso Vacância",            f"{PESO_VACANCIA*100:.0f}%"),
        ("Peso Liquidez",            f"{PESO_LIQUIDEZ*100:.0f}%"),
        ("", ""),
        ("TIPOS DE FII", ""),
        ("", ""),
        ("Tijolo",  "Imóveis físicos: logística, shopping, escritório, hospital, etc."),
        ("Papel",   "Títulos de crédito imobiliário: CRI, recebíveis"),
        ("FOF",     "Fundo de Fundos: investe em cotas de outros FIIs"),
        ("Híbrido", "Multicategoria, híbridos e outros"),
        ("", ""),
        ("AVISO LEGAL", ""),
        ("", ""),
        ("Disclaimer",
         "Esta análise é para fins informativos e educacionais. "
         "Não constitui recomendação de investimento. "
         "Consulte um profissional certificado (CFP/CNPI) antes de investir."),
    ]

    fonte_secao = Font(name='Arial', bold=True, size=11, color='1F4E79')
    fonte_normal = Font(name='Arial', size=10)

    for idx, (chave, valor) in enumerate(metadados, start=1):
        aba_meta[f'A{idx}'] = chave
        aba_meta[f'B{idx}'] = valor
        aba_meta[f'A{idx}'].font = fonte_secao if valor == "" else fonte_normal
        aba_meta[f'B{idx}'].font = fonte_normal

    aba_meta.column_dimensions['A'].width = 28
    aba_meta.column_dimensions['B'].width = 72

    # --- Salvar ---
    try:
        wb.save(nome_arquivo)
        print(f"\n✅ Planilha salva: {nome_arquivo}")
        return True
    except Exception as e:
        print(f"❌ Erro ao salvar: {e}")
        return False

In [39]:
# ===========================================================
# CÉLULA 6 — Execução principal
# ===========================================================

def executar_analise():
    """
    Orquestra todas as etapas da análise:
    1. Coleta de dados
    2. Limpeza e padronização
    3. Aplicação dos filtros de qualidade
    4. Cálculo do score e ranking
    5. Geração da planilha Excel
    6. Download automático (se no Colab)
    """

    print("=" * 55)
    print("  📊 ANÁLISE DE FIIs — CARTEIRA PREVIDENCIÁRIA")
    print("=" * 55)

    # --- ETAPA 1: Coletar dados brutos do Fundamentus ---
    df_bruto = coletar_fiis_fundamentus()

    if df_bruto is None:
        print("\n❌ Análise interrompida: falha na coleta de dados.")
        return None

    # Pausa respeitosa entre operações
    time.sleep(1)

    # --- ETAPA 2: Limpar e padronizar ---
    df_limpo = limpar_e_padronizar(df_bruto)

    if df_limpo.empty:
        print("\n❌ Análise interrompida: nenhum dado após limpeza.")
        return None

    # --- ETAPA 3: Aplicar filtros de qualidade ---
    df_filtrado = aplicar_filtros_qualidade(df_limpo)

    if df_filtrado.empty:
        print("\n⚠️ Nenhum FII passou em todos os filtros.")
        print("   Considere afrouxar os critérios nas configurações.")
        return None

    # --- ETAPA 4: Calcular scores e ranking ---
    df_rankeado = calcular_score(df_filtrado)

    # --- ETAPA 5: Gerar planilha Excel ---
    sucesso = gerar_planilha_excel(df_rankeado, NOME_ARQUIVO)

    if not sucesso:
        return None

    # --- ETAPA 6: Download automático no Colab ---
    if RODANDO_NO_COLAB:
        print(f"\n⬇️  Iniciando download da planilha...")
        files.download(NOME_ARQUIVO)

    # --- Resumo final no terminal ---
    print("\n" + "=" * 55)
    print("  ✅ ANÁLISE CONCLUÍDA COM SUCESSO")
    print("=" * 55)
    print(f"\n🏆 TOP 10 FIIs por Score:")
    print(f"{'#':<5} {'Ticker':<10} {'DY':>8} {'P/VP':>7} {'Vacância':>10} {'Score':>7}")
    print("-" * 50)

    for _, linha in df_rankeado.head(10).iterrows():
        ticker    = str(linha.get('ticker',    '-'))
        dy        = linha.get('dy_anual',   0) or 0
        pvp       = linha.get('pvp',        0) or 0
        vacancia  = linha.get('vacancia',   None)
        score     = linha.get('score_final', 0) or 0
        ranking   = int(linha.get('ranking', 0))

        vacancia_str = f"{vacancia*100:.1f}%" if pd.notna(vacancia) else "N/A"

        print(f"{ranking:<5} {ticker:<10} {dy*100:>7.2f}% {pvp:>7.2f} {vacancia_str:>10} {score:>7.2f}")

    print(f"\n📁 Arquivo gerado: {NOME_ARQUIVO}")

    return df_rankeado


# --- Ponto de entrada ---
# Esta linha chama a função principal quando o script é executado
df_resultado = executar_analise()

  📊 ANÁLISE DE FIIs — CARTEIRA PREVIDENCIÁRIA
🌐 Conectando ao Fundamentus...
✅ Conexão bem-sucedida! Extraindo tabela...
📊 559 FIIs encontrados no Fundamentus.

🔧 Limpando e padronizando os dados...
   0 FIIs removidos por dados faltantes.
   559 FIIs com dados válidos.

   Distribuição por tipo:
      Híbrido: 321 FIIs
      Tijolo: 152 FIIs
      Papel: 86 FIIs

🔍 Aplicando filtros de qualidade...
   Liquidez < R$ 500,000: 421 removidos
   P/VP fora de [0.0, 99.0]: 0 removidos
   DY < 5%: 18 removidos
   Vacância > 30%: 8 removidos

✅ 112 FIIs aprovados (de 559 analisados)

📐 Calculando scores do ranking...
✅ Score calculado para 112 FIIs.

📝 Gerando planilha Excel: analise_fiis_20260617_2006.xlsx
   ✅ Aba 'Ranking Geral': 112 FIIs
   ✅ Aba 'Tijolo': 26 FIIs
   ✅ Aba 'Papel': 6 FIIs
   ✅ Aba 'FOF': 0 FIIs
   ✅ Aba 'Híbrido': 80 FIIs

✅ Planilha salva: analise_fiis_20260617_2006.xlsx

⬇️  Iniciando download da planilha...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


  ✅ ANÁLISE CONCLUÍDA COM SUCESSO

🏆 TOP 10 FIIs por Score:
#     Ticker           DY    P/VP   Vacância   Score
--------------------------------------------------
1     TRXF11       12.73%    0.93       0.0%    7.21
2     KNCR11       13.44%    1.04       0.0%    7.15
3     HSRE11       13.29%    0.98       0.0%    6.50
4     MXRF11       13.23%    1.04        N/A    6.43
5     MANA11       27.23%    0.98       0.0%    6.33
6     GGRC11       11.96%    0.90       0.0%    6.03
7     CACR11       55.76%    0.24       0.0%    5.98
8     GARE11       12.13%    0.87       0.0%    5.98
9     KNIP11       10.48%    0.97       0.0%    5.93
10    VGIR11       15.62%    0.99       0.0%    5.93

📁 Arquivo gerado: analise_fiis_20260617_2006.xlsx
